### **SELECT THE CHOSEN JOB CODES FOR EXPERIMENT**  

Build out the logic to select the specific jobs that will use the run from the curated dataset.

INPUTS:   
  occupations_file          # raw O*NET occupations table (untouched)   
  major_groups = [11, 15, 29, 41]   # example chosen SOC major groups   
  occupations_per_group = 5        # adjustable   
  random_seed = 42                 # for reproducibility   

All random sampling operations use a fixed random seed (seed = 42) to ensure reproducibility.

FILTER occupations:  
  • keep rows where SOC_Code is a detailed occupation (6-digit).   
  • keep rows where SOC_Code starts with any major_group in major_groups. 

FOR each major_group in major_groups:  
  • subset occupations where SOC_Code starts with major_group.   
  • randomly sample occupations_per_group rows.   
  • add sampled rows to selected_occupations.   

COMBINE all sampled rows into final_selection  

SAVE final_selection to:
  data/experiment_occupations.csv  


In [23]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
import sys
import os

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add it to the system path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [25]:
import pandas as pd
import random as rnd
from pathlib import Path
from datetime import datetime
import src.utils.functions as utils

In [26]:
PROJECT_ROOT = utils.find_project_root()
(PROJECT_ROOT / "data").exists()

True

In [ ]:
# SET UP PATHS TO FILES AND DIRECTORIES

#--- DIRECTORIES ---#
input_dir = PROJECT_ROOT / "data/onet_datasets/curated/"
output_dir = PROJECT_ROOT / "data/onet_datasets/experiment_datasets"

#---- FILE ---#
input_file = input_dir/'onet_curated_dataset.csv'

In [28]:
with open(input_dir / "raw/Occupation_Data.txt") as f:
    header = f.readline().strip()

columns = header.split("\t") 
print(columns)
occupation_df = pd.read_csv(
    input_dir / "raw/Occupation_Data.txt",
    sep="\t",
    skiprows=1,  # skip the first (comma) header line
    names=columns,
    encoding="utf-8"
)
occupation_df.columns = occupation_df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_").str.replace("*","")

['O*NET-SOC Code', 'Title', 'Description']


Select the first K major groups present in the dataset after sorting by SOC code.

In [ ]:
#---- Extract the major groups for SOC_Codes ----#

dataset = utils.load_csv(input_file, separator=',')

def get_k_random_soc_codes(df, k, major_group=None, seed=42):
    """
    Returns a list of K unique detailed SOC codes.
    If major_group is provided, sampling is restricted to that 2-digit group.
    """

    rnd.seed(seed)
    
    # Use a single, consistent SOC column
    soc_col = "Job_Code"

    # Optional major group filter
    if major_group is not None:
        mask = df[soc_col].str.startswith(str(major_group))
        candidate_codes = df[mask][soc_col].unique()
    else:
        candidate_codes = df[soc_col].unique()

    # Safety check
    if k > len(candidate_codes):
        print(
            f"Warning: Requested {k} codes, but only {len(candidate_codes)} available."
        )
        k = len(candidate_codes)

    # Random sampling
    sampled_codes = rnd.sample(list(candidate_codes), k)

    return sampled_codes

In [30]:
# --- RUNNING THE SELECTION ---
my_seed = 42

# Get 5 completely random codes
selected_codes = get_k_random_soc_codes(dataset, k=5, seed=my_seed)
print("Selected SOC Codes:", selected_codes)

Selected SOC Codes: ['47-5023.00', '15-2051.02', '11-9033.00', '51-6051.00', '25-2021.00']


In [31]:
def log_prompt_occupations(
    occupation_df,
    selected_codes,
    strategy_name,
    seed=None,
    experiment_id=None
):
    prompt_df = occupation_df[
        occupation_df["ONET_SOC_Code"].isin(selected_codes)
    ][["ONET_SOC_Code", "Title"]].drop_duplicates().copy()

    prompt_df["Major_Group"] = prompt_df["ONET_SOC_Code"].str[:2]
    prompt_df["Selection_Strategy"] = strategy_name
    prompt_df["Selection_Seed"] = seed

    if experiment_id is None:
        experiment_id = f"EXP_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    prompt_df["Experiment_ID"] = experiment_id

    assert prompt_df["ONET_SOC_Code"].is_unique

    return prompt_df

# Log the selection
experiment_log = log_prompt_occupations(
    occupation_df, 
    selected_codes, 
    strategy_name="random_sample_per_major_group", 
    seed=my_seed,
    experiment_id = "rav_test_001"
)

# 3. Save to CSV
experiment_log.to_csv(output_dir / 'test_occupations_list.csv', index=False)
print(f"Logged {len(experiment_log)} jobs to experiment_selections.csv")

Logged 5 jobs to experiment_selections.csv


In [32]:
#--- CREATE EXPERIMENT CSV ---
def log_experiment_selection(df, selected_codes, strategy_name, seed=None,experiment_id=None):
    """
    Creates a dataframe of the selected jobs for experiment.
    """
    # 1. Filter the master dataframe for only the selected codes
    selection_df = df[df['Job_Code'].isin(selected_codes)].copy()
    
    # 2. Add the Metadata
    selection_df['Major_Group'] = selection_df['Job_Code'].str[:2]
    selection_df['Selection_Seed'] = seed
    selection_df['Selection_Strategy'] = strategy_name
    
    # Create a unique ID based on timestamp
    experiment_id = f"EXP_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    selection_df['Experiment_ID'] = experiment_id
    
    return selection_df

# --- RUNNING THE SELECTION ---
my_seed = 42

# Get 5 completely random codes
selected_codes = get_k_random_soc_codes(dataset, k=5, seed=my_seed)
print("Selected SOC Codes:", selected_codes)

# Log the selection
experiment_log = log_experiment_selection(
    dataset, 
    selected_codes, 
    strategy_name="random_sample_per_major_group", 
    seed=my_seed,
    experiment_id = "rav_test_001"
)

# 3. Save to CSV
experiment_log.to_csv(output_dir / 'test_KG_selection.csv', index=False)
print(f"Logged {len(experiment_log)} jobs to test_experiment_KG_selection.csv")

Selected SOC Codes: ['47-5023.00', '15-2051.02', '11-9033.00', '51-6051.00', '25-2021.00']
Logged 540 jobs to test_experiment_KG_selection.csv
